In [5]:
import os
import kagglehub
import shutil

# 1. Download the dataset using kagglehub
print("Downloading/Locating dataset...")
path = kagglehub.dataset_download("jangedoo/utkface-new")
print(f"Path to dataset files: {path}")

# 2. Define your target destination
target_data_dir = '/content/utk_data'

# 3. Copy files to a writable directory if not already there
if not os.path.exists(target_data_dir):
    os.makedirs(target_data_dir)

    # Assuming UTKFace images are in a subfolder or directly in the path
    # Adjust the source path if needed (sometimes it is inside 'UTKFace' folder)
    source_path = os.path.join(path, 'UTKFace')

    # If UTKFace folder doesn't exist, try searching the root of 'path'
    if not os.path.exists(source_path):
        source_path = path

    files = [f for f in os.listdir(source_path) if f.endswith('.jpg')]
    for file in files:
        shutil.copy(os.path.join(source_path, file), os.path.join(target_data_dir, file))
    print(f"✅ Copied {len(files)} images to {target_data_dir}")
else:
    print(f"✅ {target_data_dir} already exists.")

# 4. NOW run your sampling code
# Verify the count before sampling
all_images = [f for f in os.listdir(target_data_dir) if f.endswith('.jpg')]
if len(all_images) >= 5000:
    selected_images = random.sample(all_images, 5000)
    print(f"Successfully selected {len(selected_images)} images.")
else:
    print(f"❌ Error: Only {len(all_images)} images available. Please select a smaller sample size.")

Downloading/Locating dataset...
Using Colab cache for faster access to the 'utkface-new' dataset.
Path to dataset files: /kaggle/input/utkface-new
✅ Copied 23708 images to /content/utk_data
Successfully selected 5000 images.


In [10]:
import os
import pandas as pd
import random
import shutil

# 1. Setup paths
source_dir = '/content/utk_data'  # Ensure this contains your images
target_dir = '/content/master_nat_age'

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# 2. Define labels by parsing filenames
race_to_nationality = {0: 'American', 1: 'African', 2: 'Others', 3: 'Indian', 4: 'Others'}
data = []

# Assuming filenames are in format: age_gender_race_timestamp.jpg
for filename in os.listdir(source_dir):
    if filename.endswith(".jpg"):
        try:
            parts = filename.split('_')
            age = int(parts[0])
            gender = int(parts[1])
            race = int(parts[2])
            nationality = race_to_nationality.get(race, 'Others')

            data.append({
                'file_path': os.path.join(source_dir, filename),
                'age': age,
                'nationality': nationality,
                'filename': filename
            })
        except (ValueError, IndexError):
            continue

# Create the full DataFrame (this resolves the NameError)
df = pd.DataFrame(data)

# 3. Sample 5000 images
if len(df) < 5000:
    print(f"Error: Dataset only has {len(df)} images. Cannot sample 5000.")
else:
    # Get random sample of filenames
    selected_samples = df.sample(n=5000, random_state=42)

    # 4. Copy files
    print(f"Copying 5000 images to {target_dir}...")
    for idx, row in selected_samples.iterrows():
        shutil.copy(row['file_path'], os.path.join(target_dir, row['filename']))

    # 5. Create final CSV for the subset
    # Update paths to point to the new location
    selected_samples['file_path'] = target_dir + '/' + selected_samples['filename']
    selected_samples.to_csv('master_nat_age_labels.csv', index=False)

    print("✅ Successfully created 'master_nat_age_labels.csv' and copied images.")

Copying 5000 images to /content/master_nat_age...
✅ Successfully created 'master_nat_age_labels.csv' and copied images.


In [14]:
import shutil
import os

folder_path = '/content/master_nat_age'

if os.path.exists(folder_path):
    try:
        shutil.rmtree(folder_path)
        print(f"✅ Folder '{folder_path}' deleted successfully.")
    except OSError as e:
        print(f"❌ Error: {e.strerror}")
else:
    print(f"⚠️ Folder '{folder_path}' does not exist.")

✅ Folder '/content/dataset_split' deleted successfully.


In [11]:
import os

folder_path = '/content/master_nat_age'
# Define the image extensions you are looking for
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def count_images(path):
    if not os.path.exists(path):
        return f"Folder '{path}' does not exist."

    count = 0
    for filename in os.listdir(path):
        if filename.lower().endswith(valid_extensions):
            count += 1
    return count

total_images = count_images(folder_path)
print(f"✅ There are {total_images} images in {folder_path}.")

✅ There are 5000 images in /content/master_nat_age.


In [13]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Setup paths
source_dir = '/content/master_nat_age'
base_output_dir = '/content/net_age_split'
splits = ['train', 'test', 'val']

# 2. Create directory structure
for split in splits:
    os.makedirs(os.path.join(base_output_dir, split), exist_ok=True)

# 3. Load your master metadata
df = pd.read_csv('master_nat_age_labels.csv')

# 4. Perform the splits
# First split: Train (70%) and Remaining (30%)
train_df, rem_df = train_test_split(df, test_size=0.3, random_state=42)

# Second split: Test (15% of total) and Val (15% of total)
test_df, val_df = train_test_split(rem_df, test_size=0.5, random_state=42)

# 5. Copy files and save CSVs
split_dfs = {'train': train_df, 'test': test_df, 'val': val_df}

for split_name, split_df in split_dfs.items():
    print(f"Processing {split_name} set ({len(split_df)} images)...")

    # Copy files
    for idx, row in split_df.iterrows():
        src = row['file_path'] # Original path in master_nat_age
        dst = os.path.join(base_output_dir, split_name, row['filename'])
        shutil.copy(src, dst)

    # Update path in dataframe to reflect new location
    split_df = split_df.copy()
    split_df['file_path'] = os.path.join(base_output_dir, split_name) + '/' + split_df['filename']

    # Save CSV
    csv_name = f'nat_age_{split_name}.csv'
    split_df.to_csv(csv_name, index=False)
    print(f"✅ Created {csv_name}")

print("Dataset distribution complete!")

Processing train set (3500 images)...
✅ Created nat_age_train.csv
Processing test set (750 images)...
✅ Created nat_age_test.csv
Processing val set (750 images)...
✅ Created nat_age_val.csv
Dataset distribution complete!


In [35]:
c

✅ Folder '/content/sanity_check' deleted successfully.


In [20]:
import os

# Set your folder path
folder_path = '/content/net_age_split' # Ensure this points to your actual dataset folder
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def count_images_recursive(path):
    if not os.path.exists(path):
        return f"❌ Folder '{path}' does not exist."

    # Initialize counts for each split
    counts = {'total': 0, 'train': 0, 'test': 0, 'val': 0}

    for root, dirs, files in os.walk(path):
        for file in files:
            if file.lower().endswith(valid_extensions):
                counts['total'] += 1

                # Assign to specific splits based on folder name
                if 'train' in root:
                    counts['train'] += 1
                elif 'test' in root:
                    counts['test'] += 1
                elif 'val' in root:
                    counts['val'] += 1

    return counts

# Run the count
counts = count_images_recursive(folder_path)

if isinstance(counts, dict):
    print(f"✅ Total images found: {counts['total']}")
    print(f"   - Train: {counts['train']}")
    print(f"   - Test: {counts['test']}")
    print(f"   - Val: {counts['val']}")
else:
    print(counts)

✅ Total images found: 5000
   - Train: 3500
   - Test: 750
   - Val: 750


# Custom Model

In [50]:
class NationalityDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        self.label_map = {'American': 0, 'African': 1, 'Indian': 2, 'Others': 3}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')

        # Nationality (Categorical)
        nat_label = self.label_map.get(row['nationality'], 3)

        # Age (Continuous/Regression)
        age_label = float(row['age'])

        if self.transform:
            image = self.transform(image)

        return image, nat_label, torch.tensor(age_label, dtype=torch.float32)

# Re-create your loaders with the updated class
train_dataset = NationalityDataset('nat_age_train.csv', transform=transform)
val_dataset = NationalityDataset('nat_age_val.csv', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# Re-create the test dataset and loader with the updated class
test_dataset = NationalityDataset('nat_age_test.csv', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ train_test_val_loader updated successfully to support Nationality and Age.")

✅ train_test_val_loader updated successfully to support Nationality and Age.


In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskCNN(nn.Module):
    def __init__(self, num_nationality_classes=4):
        super(MultiTaskCNN, self).__init__()

        # --- Shared Feature Extraction (3 Layers) ---
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # --- Head 1: Nationality Classification (2 layers) ---
        self.nat_fc1 = nn.Linear(128 * 16 * 16, 256)
        self.nat_fc2 = nn.Linear(256, num_nationality_classes)

        # --- Head 2: Age Regression (2 layers) ---
        self.age_fc1 = nn.Linear(128 * 16 * 16, 256)
        self.age_fc2 = nn.Linear(256, 1) # Single output for Age

    def forward(self, x):
        # Shared Convolutional Base
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten for the heads
        x = x.view(x.size(0), -1)

        # Branch 1: Nationality
        nat_out = F.relu(self.nat_fc1(x))
        nat_out = self.nat_fc2(nat_out)

        # Branch 2: Age
        age_out = F.relu(self.age_fc1(x))
        age_out = self.age_fc2(age_out)

        return nat_out, age_out

# Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskCNN().to(device)
print(f"✅ Multi-task model initialized on {device}")

✅ Multi-task model initialized on cpu


In [ ]:
import torch.optim as optim

# 1. Initialize history storage
history = {'nat_loss': [], 'age_loss': [], 'total_loss': []}

# Loss functions and Optimizer
nat_criterion = nn.CrossEntropyLoss()
age_criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_multitask(num_epochs=10):
    print("Starting training...")
    for epoch in range(num_epochs):
        model.train()
        # Initialize running metrics for this epoch
        running_total_loss = 0.0
        running_nat_loss = 0.0
        running_age_loss = 0.0

        for images, nat_labels, age_labels in train_loader:
            images, nat_labels, age_labels = images.to(device), nat_labels.to(device), age_labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            nat_pred, age_pred = model(images)

            # Separate components for history logging
            loss_nat = nat_criterion(nat_pred, nat_labels)
            loss_age = age_criterion(age_pred.squeeze(), age_labels)

            # Combined Loss
            loss = loss_nat + (0.1 * loss_age)

            loss.backward()
            optimizer.step()

            # Accumulate metrics
            running_total_loss += loss.item()
            running_nat_loss += loss_nat.item()
            running_age_loss += loss_age.item()

        # Calculate averages for the epoch
        avg_total = running_total_loss / len(train_loader)
        avg_nat = running_nat_loss / len(train_loader)
        avg_age = running_age_loss / len(train_loader)

        # Save to history
        history['total_loss'].append(avg_total)
        history['nat_loss'].append(avg_nat)
        history['age_loss'].append(avg_age)

        print(f"Epoch {epoch+1}/{num_epochs} | Total Loss: {avg_total:.4f} | "
              f"Nat Loss: {avg_nat:.4f} | Age Loss: {avg_age:.4f}")

    # Save the model
    torch.save(model.state_dict(), 'multitask_model.pth')
    print("✅ Training complete. Model saved as 'multitask_model.pth'")

# Run the training
train_multitask(num_epochs=10)

In [ ]:
import matplotlib.pyplot as plt

def plot_all_histories(history):
    # Create a figure with a grid layout
    # 2 rows, 2 columns.
    # Row 1 will span both columns for the Dual-Axis Plot
    # Row 2 will have the individual plots
    fig = plt.figure(figsize=(14, 10))

    # 1. Dual-Axis Plot (Top Row, spanning 2 columns)
    ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Nationality Loss (CrossEntropy)', color='tab:blue')
    ax1.plot(history['nat_loss'], color='tab:blue', label='Nationality Loss', marker='o')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Age Loss (MSE)', color='tab:red')
    ax2.plot(history['age_loss'], color='tab:red', label='Age Loss', marker='s')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    ax2.set_title('Multi-Task Training Loss over Epochs (Dual-Axis)')

    # 2. Nationality Loss (Bottom Left)
    ax3 = plt.subplot2grid((2, 2), (1, 0))
    ax3.plot(history['nat_loss'], color='tab:blue', marker='o', label='Nat Loss')
    ax3.set_title('Nationality Classification Loss')
    ax3.set_xlabel('Epochs'); ax3.set_ylabel('CrossEntropy')
    ax3.grid(True, linestyle='--', alpha=0.6)

    # 3. Age Loss (Bottom Right)
    ax4 = plt.subplot2grid((2, 2), (1, 1))
    ax4.plot(history['age_loss'], color='tab:red', marker='s', label='Age Loss')
    ax4.set_title('Age Regression Loss')
    ax4.set_xlabel('Epochs'); ax4.set_ylabel('MSE')
    ax4.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# Call this after your training loop
plot_all_histories(history)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

def evaluate_model(model, test_loader, device):
    model.eval()
    all_nat_preds = []
    all_nat_targets = []
    all_age_preds = []
    all_age_targets = []

    with torch.no_grad():
        for images, nat_labels, age_labels in test_loader:
            images = images.to(device)
            nat_pred, age_pred = model(images)

            # Nationality Predictions
            _, predicted_nat = torch.max(nat_pred, 1)
            all_nat_preds.extend(predicted_nat.cpu().numpy())
            all_nat_targets.extend(nat_labels.numpy())

            # Age Predictions
            all_age_preds.extend(age_pred.squeeze().cpu().numpy())
            all_age_targets.extend(age_labels.numpy())

    # 1. Confusion Matrix for Nationality
    cm = confusion_matrix(all_nat_targets, all_nat_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['American', 'African', 'Indian', 'Others'],
                yticklabels=['American', 'African', 'Indian', 'Others'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Nationality Confusion Matrix')
    plt.show()

    # 2. Accuracy Visualization
    acc = accuracy_score(all_nat_targets, all_nat_preds)
    print(f"✅ Nationality Classification Accuracy: {acc*100:.2f}%")

    # 3. Age Prediction Visualization (Predicted vs Actual)
    plt.figure(figsize=(8, 5))
    plt.scatter(all_age_targets, all_age_preds, alpha=0.5)
    plt.plot([min(all_age_targets), max(all_age_targets)], [min(all_age_targets), max(all_age_targets)], color='red')
    plt.xlabel('Actual Age')
    plt.ylabel('Predicted Age')
    plt.title('Age Regression: Predicted vs Actual')
    plt.show()

# Execute evaluation
evaluate_model(model, test_loader, device)

# Emotion Identification

In [53]:
import kagglehub
import os
import pandas as pd
import glob

# Download
fer_path = kagglehub.dataset_download("msambare/fer2013")
print(f"FER-2013 is at: {fer_path}")

Using Colab cache for faster access to the 'fer2013' dataset.
FER-2013 is at: /kaggle/input/fer2013


In [63]:
import os
import shutil
import random

source_root = '/kaggle/input/fer2013'
target_root = 'master_emotion'

# Define our sampling targets
# Ensure the structure exists in source as: source_root/train/emotion_name/image.jpg
splits = {'train': 4000, 'test': 1000}

for split, total_limit in splits.items():
    split_path = os.path.join(source_root, split)

    # 1. Collect all valid images grouped by their emotion subfolder
    files_by_category = {}
    for category in os.listdir(split_path):
        cat_path = os.path.join(split_path, category)
        if os.path.isdir(cat_path):
            # Gather all files in this emotion subfolder
            files_by_category[category] = [
                os.path.join(cat_path, f) for f in os.listdir(cat_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]

    # 2. Flatten all files while keeping their category label
    all_files_with_cat = []
    for cat, files in files_by_category.items():
        for f in files:
            all_files_with_cat.append((f, cat))

    # 3. Shuffle and take only the required limit for this split
    random.shuffle(all_files_with_cat)
    selected_files = all_files_with_cat[:total_limit]

    print(f"✅ Processing {split}: Selected {len(selected_files)} images.")

    # 4. Copy files to target_root/category/filename
    for file_path, category in selected_files:
        # Create the emotion subfolder in the destination
        dest_dir = os.path.join(target_root, category)
        os.makedirs(dest_dir, exist_ok=True)

        # Copy the file
        shutil.copy2(file_path, os.path.join(dest_dir, os.path.basename(file_path)))

print(f"✅ Finished! Dataset created at: {target_root}")

✅ Processing train: Selected 4000 images.
✅ Processing test: Selected 1000 images.
✅ Finished! Dataset created at: master_emotion


In [65]:
import os

folder_path = '/content/master_emotion'
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def count_images_recursively(path):
    if not os.path.exists(path):
        return f"Folder '{path}' does not exist."

    count = 0
    # os.walk visits every subdirectory
    for root, dirs, files in os.walk(path):
        for filename in files:
            if filename.lower().endswith(valid_extensions):
                count += 1
    return count

total_images = count_images_recursively(folder_path)
print(f"✅ There are {total_images} images in {folder_path} and its subfolders.")

✅ There are 5000 images in /content/master_emotion and its subfolders.


In [66]:
import os
import pandas as pd

def create_master_csv(root_dir, output_csv):
    data = []
    # List of valid extensions to check against
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

    # Walk through the directory tree
    for root, dirs, files in os.walk(root_dir):
        for filename in files:
            if filename.lower().endswith(valid_extensions):
                # The parent folder name is the emotion label
                label = os.path.basename(root)
                # Store the file path relative to the root or just the filename
                data.append({'filename': filename, 'label': label})

    # Convert to DataFrame and save
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"✅ CSV created successfully with {len(df)} rows at '{output_csv}'.")

# Run the function
create_master_csv('/content/master_emotion', 'master_emotion.csv')

✅ CSV created successfully with 5000 rows at 'master_emotion.csv'.


In [68]:
!pip install split-folders

In [69]:
import splitfolders
import os
import pandas as pd

# 1. Perform the split
input_folder = 'master_emotion'
output_folder = 'emotion_split'

# Splits into 80% train, 10% val, 10% test
splitfolders.ratio(input_folder, output=output_folder,
                   seed=42, ratio=(0.8, 0.1, 0.1),
                   group_prefix=None, move=False)

def generate_split_csvs(base_path):
    splits = ['train', 'val', 'test']

    for split in splits:
        split_path = os.path.join(base_path, split)
        data = []

        # Walk through the specific split folder
        for root, dirs, files in os.walk(split_path):
            for filename in files:
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    # Extract emotion label from folder name
                    label = os.path.basename(root)
                    data.append({'filename': filename, 'label': label})
        print("")
        # Save CSV
        df = pd.DataFrame(data)
        csv_name = f"{split}_emotion.csv"
        df.to_csv(csv_name, index=False)
        print(f"✅ Generated {csv_name} with {len(df)} images.")

# Run the generation
generate_split_csvs(output_folder)

Copying files: 5000 files [00:00, 5843.51 files/s]

✅ Generated train_emotion.csv with 3997 images.
✅ Generated val_emotion.csv with 496 images.
✅ Generated test_emotion.csv with 507 images.


# Custom Model

In [111]:
import torch
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset

class EmotionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file.
            root_dir (string): Directory with all the images (e.g., 'emotion_split/train').
            transform (callable, optional): Optional transform to be applied.
        """
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.label_map = {
            'angry': 0, 'disgust': 1, 'fear': 2,
            'happy': 3, 'sad': 4, 'surprise': 5, 'neutral': 6
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Corrected: Build the full path using root_dir and the label/filename
        # Assuming your structure is root_dir/label/filename
        img_path = os.path.join(self.root_dir, row['label'], row['filename'])

        # FER2013 is grayscale, use 'L'
        image = Image.open(img_path).convert('L')

        # Get label
        emotion_label = self.label_map.get(row['label'], 6)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(emotion_label, dtype=torch.long)

In [122]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

class EmotionCNN(nn.Module):
    def __init__(self):
        super(EmotionCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)

        # Block 3
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # After 3 pooling layers, 48x48 becomes 6x6
        # Calculation: 48 -> 24 (pool1) -> 12 (pool2) -> 6 (pool3)
        self.fc1 = nn.Linear(256 * 6 * 6, 512)
        self.fc2 = nn.Linear(512, 7)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = F.relu(self.bn4(self.conv4(x))) # Added intermediate conv without pool
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))

        x = x.view(-1, 256 * 6 * 6)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [123]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize Model, Loss, and Optimizer
model = EmotionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 3. Setup Transformations
transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 4. Initialize DataLoaders
# Pointing to your 'sanity_emotion' (or full 'emotion_split')
train_dataset = EmotionDataset('train_emotion.csv', 'emotion_split/train', transform=transform)
val_dataset = EmotionDataset('val_emotion.csv', 'emotion_split/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [115]:
import torch
import numpy as np
from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20, device='cuda', patience=5):
    model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        history['train_loss'].append(running_loss / len(train_loader))
        history['train_acc'].append(100 * correct / total)

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}: Train Loss: {history['train_loss'][-1]:.4f} | Train Acc: {history['train_acc'][-1]:.2f}% | Val Loss: {history['val_loss'][-1]:.4f} | Val Acc: {val_acc:.2f}%")

        # --- Early Stopping & Checkpointing ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_emotion_model.pth')
            print("✅ Model saved (Improved accuracy)")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⚠️ Early stopping at epoch {epoch+1}")
                break

    return history

In [116]:
import matplotlib.pyplot as plt

def plot_history(history):
    plt.figure(figsize=(12, 4))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss over Epochs')
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Accuracy over Epochs')
    plt.legend()

    plt.show()

In [ ]:
# Run the training process
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=20,
    device=device,
    patience=5
)

# Optional: Visualize results after training
plot_history(history)

In [ ]:
import torch
from torch.utils.data import DataLoader

def test_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds

# 1. Setup Test Dataset and Loader
# Ensure you use the actual test set from your full dataset, not the sanity one
test_dataset = EmotionDataset('test_emotion.csv', 'emotion_split/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Load the Best Weights
model = EmotionCNN().to(device)
model.load_state_dict(torch.load('best_emotion_model.pth'))

# 3. Execute Testing
y_true, y_pred = test_model(model, test_loader, device)
print("✅ Testing complete.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_test_results(y_true, y_pred):
    emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

    # 1. Classification Report (Text-based accuracy)
    print("Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=emotion_labels))

    # 2. Confusion Matrix Heatmap
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=emotion_labels,
                yticklabels=emotion_labels)
    plt.title('Confusion Matrix: Emotion Identification')
    plt.xlabel('Predicted Emotion')
    plt.ylabel('Actual Emotion')
    plt.show()

# Trigger the visualization
visualize_test_results(y_true, y_pred)

# **Github Repo**

https://github.com/KVAlwaysLearning/Nationality_Detection_Sub

# **Streamlit-app**

https://nationalitydetectionsub-app.streamlit.app/